In [2]:
from utils.sampler import sample_from_model
import os
import torch

from model import GPTConfig, GPT

In [3]:
from eval.evaluation_pipeline.lm_eval.api.model import LM

ModuleNotFoundError: No module named 'lm_eval'

In [16]:
out_dir = 'output_dump/out-babylm_full_bpe-4x4-nomask-1709506109' # ignored if init_from is not 'resume'
data_dir = os.path.join('data','babylm_full_bpe')
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
device = 'cuda'
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
print("G2",gptconf)
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.to(device)



G2 GPTConfig(block_size=128, vocab_size=16000, n_layer=4, n_head=4, n_embd=256, dropout=0.1, bias=False, wm_mask=False, wm_decay_rate=1, wm_decay_type='linear')
number of parameters: 7.24M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(16000, 256)
    (wpe): Embedding(128, 256)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=256, out_features=768, bias=False)
          (c_proj): Linear(in_features=256, out_features=256, bias=False)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=256, out_features=1024, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1024, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=256, out_features=16000, bias=False)
)

In [18]:
print(device)

import time

start = time.time()

sample_from_model(model, data_dir, out_dir, dtype, device=device, prompt_type="elaborate")

print("Time taken: ", time.time()-start)

cuda
Time taken:  15.859974384307861


In [11]:
from tokenizers.normalizers import Lowercase, Strip, StripAccents, NFD
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer)

In [10]:
tokenizer = Tokenizer(models.BPE())
normalizer = normalizers.Sequence([NFD(), Lowercase(), Strip(), StripAccents()])
tokenizer.normalizer = normalizer
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(vocab_size=16000, special_tokens=["<|endoftext|>"])
path_to_data = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/babylm_data'
textfiles = [f"{path_to_data}/babylm_10M/aochildes.train",
             f"{path_to_data}/babylm_10M/bnc_spoken.train",
             f"{path_to_data}/babylm_10M/cbt.train",
             f"{path_to_data}/babylm_10M/children_stories.train",
             f"{path_to_data}/babylm_10M/gutenberg.train",
             f"{path_to_data}/babylm_10M/open_subtitles.train",
             f"{path_to_data}/babylm_10M/qed.train",
             f"{path_to_data}/babylm_10M/simple_wikipedia.train",
             f"{path_to_data}/babylm_10M/switchboard.train",
             f"{path_to_data}/babylm_10M/wikipedia.train"]

tokenizer.train(files = textfiles, trainer=trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
tokenizer.decoder = decoders.ByteLevel()


In [5]:
string_list = ["Let's test this tokenizer.", "This is a test string.", "This is another test string." ]

In [6]:
from transformers import AutoTokenizer

data_dir = os.path.join('data','babylm_full_bpe')
tokenizer2 = AutoTokenizer.from_pretrained(data_dir, use_fast=False)


if not tokenizer2.pad_token:
    tokenizer2.pad_token = tokenizer2.eos_token
tokenizer2.padding_side = "left"
tokenizer2(string_list, padding=True, return_tensors="pt")

{'input_ids': tensor([[ 683,  225, 1910,  287,  196, 2337, 9961,   14],
        [   0,    0,  633,  229,  171, 1910, 6044,   14],
        [   0,    0,  633,  229,  928, 1910, 6044,   14]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1]])}

In [44]:

tokenizer2.encode("<|endoftext|>", add_special_tokens=False)


[0]

In [22]:
ix,_ = model(x)[0]
print(ix)

ValueError: not enough values to unpack (expected 2, got 1)

In [4]:
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.ids)

print(tokenizer.decode([3456,7673,1111,7777]))

[683, 225, 1910, 287, 196, 2337, 9961, 14]
father documentsret sel


In [14]:
encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [14]:
out_dir = 'output_dump/out-babylm_full_bpe-4x4-nomask-1709506109' # ignored if init_from is not 'resume'
data_dir = os.path.join('data','babylm_full_bpe')
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
device = 'cuda'
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
print("G2",gptconf)
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.to(device)


G2 GPTConfig(block_size=128, vocab_size=16000, n_layer=4, n_head=4, n_embd=256, dropout=0.1, bias=False, wm_mask=False, wm_decay_rate=1, wm_decay_type='linear')
number of parameters: 7.24M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(16000, 256)
    (wpe): Embedding(128, 256)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=256, out_features=768, bias=False)
          (c_proj): Linear(in_features=256, out_features=256, bias=False)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=256, out_features=1024, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1024, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=256, out_features=16000, bias=False)
)

In [52]:



x = (torch.tensor(encoding.ids, dtype=torch.long, device=device)[None, ...])
print(x)


tensor([[ 683,  225, 1910,  287,  196, 2337, 9961,   14]], device='cuda:0')


In [27]:
ix,_ = model(x)
print(ix)

tensor([[[-16.7433,  -1.5512,   1.8630,  ...,  -8.6782,  -7.1254,  -8.1905]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)


In [7]:
import torch.nn.functional as F

encoding2 = tokenizer2(["Let's test this tokenizer.", "This is a test string.", "This is another test string."], padding=True, return_tensors="pt")
print(encoding2.input_ids)




tensor([[ 683,  225, 1910,  287,  196, 2337, 9961,   14],
        [   0,    0,  633,  229,  171, 1910, 6044,   14],
        [   0,    0,  633,  229,  928, 1910, 6044,   14]])


In [8]:
F.log_softmax(encoding2)

AttributeError: 

In [9]:
model.generate(torch.tensor(encoding2.input_ids, dtype=torch.long, device=device), max_new_tokens=10)

/tmp/ipykernel_289393/47476823.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  model.generate(torch.tensor(encoding2.input_ids, dtype=torch.long, device=device), max_new_tokens=10)


tensor([[  683,   225,  1910,   287,   196,  2337,  9961,    14,   262,   287,
           229,  7205,   207,   222,  1953,   381,   318,   198],
        [    0,     0,   633,   229,   171,  1910,  6044,    14,   133,   302,
           292,   287,  1288,   203,  5983,    14,   133,   251],
        [    0,     0,   633,   229,   928,  1910,  6044,    14,   198,  1121,
           196,   223,   412,  1165,  7800, 15956,    14,   198]],
       device='cuda:0')

In [67]:
from transformers import AutoConfig, AutoModelForCausalLM

# Download configuration from huggingface.co and cache.
config = AutoConfig.from_pretrained("google-bert/bert-base-cased")
model2 = AutoModelForCausalLM.from_config(config)
model2.to(device)

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


BertLMHeadModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [10]:
model2.generate(torch.tensor(encoding2.input_ids, dtype=torch.long, device=device), max_new_tokens=10, do_sample=False)

NameError: name 'model2' is not defined

In [11]:
encoding3 = tokenizer2(["Let's test this tokenizer.", "This is a test string.", "This is another test string."], padding=True, return_tensors="pt")
encoding4 = tokenizer2(["Let test this tokenizer.", "This is a test string.", "This is another test string."], padding=True, return_tensors="pt")
encoding5 = tokenizer2(["This is a test string.", "This is another test string."], padding=True, return_tensors="pt")

print(encoding3.input_ids, "encoding3")
print(encoding4.input_ids, "encoding4")
print(encoding5.input_ids, "encoding5")

tensor3 = torch.tensor(encoding3.input_ids, dtype=torch.long, device=device)
tensor4 = torch.tensor(encoding4.input_ids, dtype=torch.long, device=device)
tensor5 = torch.tensor(encoding5.input_ids, dtype=torch.long, device=device)


tensor([[ 683,  225, 1910,  287,  196, 2337, 9961,   14],
        [   0,    0,  633,  229,  171, 1910, 6044,   14],
        [   0,    0,  633,  229,  928, 1910, 6044,   14]]) encoding3
tensor([[ 683, 1910,  287,  196, 2337, 9961,   14],
        [   0,  633,  229,  171, 1910, 6044,   14],
        [   0,  633,  229,  928, 1910, 6044,   14]]) encoding4
tensor([[ 633,  229,  171, 1910, 6044,   14],
        [ 633,  229,  928, 1910, 6044,   14]]) encoding5


/tmp/ipykernel_289393/2050004470.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor3 = torch.tensor(encoding3.input_ids, dtype=torch.long, device=device)
/tmp/ipykernel_289393/2050004470.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor4 = torch.tensor(encoding4.input_ids, dtype=torch.long, device=device)
/tmp/ipykernel_289393/2050004470.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor5 = torch.tensor(encoding5.input_ids, dtype=torch.long, device=device)


In [84]:
out3 = model.generate(tensor3, 2, do_sample=False)
out4 = model.generate(tensor4, 2, do_sample=False)
out5 = model.generate(tensor5, 2, do_sample=False)

print(out3, "out3")
print(out4, "out4")
print(out5, "out5")

test1
test1
test1
test1
test1
test1
tensor([[ 683,  225, 1910,  287,  196, 2337, 9961,   14,  133,   47],
        [   0,    0,  633,  229,  171, 1910, 6044,   14,  133,   47],
        [   0,    0,  633,  229,  928, 1910, 6044,   14,  133,   47]],
       device='cuda:0') out3
tensor([[ 683, 1910,  287,  196, 2337, 9961,   14,  133,   47],
        [   0,  633,  229,  171, 1910, 6044,   14,  133,  199],
        [   0,  633,  229,  928, 1910, 6044,   14,  133,  251]],
       device='cuda:0') out4
tensor([[ 633,  229,  171, 1910, 6044,   14,  133,  199],
        [ 633,  229,  928, 1910, 6044,   14,  133,   47]], device='cuda:0') out5


In [71]:
import transformers

class MultiTokenEOSCriteria(transformers.StoppingCriteria):
    """Criteria to stop on the specified multi-token sequence."""

    def __init__(
        self,
        sequence: str,
        tokenizer,
        initial_decoder_input_length: int,
        batch_size: int,
    ):
        self.initial_decoder_input_length = initial_decoder_input_length
        self.done_tracker = [False] * batch_size
        self.sequence = sequence
        self.sequence_ids = tokenizer.encode(sequence, add_special_tokens=False)
        self.sequence_id_len = len(self.sequence_ids)
        self.tokenizer = tokenizer

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        # For efficiency, we compare the last n tokens where n is the number of tokens in the stop_sequence
        lookback_ids_batch = input_ids[:, self.initial_decoder_input_length :][
            :, -self.sequence_id_len :
        ]

        lookback_tokens_batch = self.tokenizer.batch_decode(lookback_ids_batch)

        for i, done in enumerate(self.done_tracker):
            if not done:
                self.done_tracker[i] = self.sequence in lookback_tokens_batch[i]
        return False not in self.done_tracker


def stop_sequences_criteria(
    tokenizer,
    stop_sequences,
    initial_decoder_input_length: int,
    batch_size: int,
) -> transformers.StoppingCriteriaList:
    return transformers.StoppingCriteriaList(
        [
            *[
                MultiTokenEOSCriteria(
                    sequence, tokenizer, initial_decoder_input_length, batch_size
                )
                for sequence in stop_sequences
            ],
        ]
    )

stop = ["A"]
input_ids = torch.tensor(encoding2.input_ids, dtype=torch.long, device=device)
stopping_criteria = stop_sequences_criteria(tokenizer2, stop, input_ids.shape[1], input_ids.shape[0]
        )

/tmp/ipykernel_40080/3728733118.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(encoding2.input_ids, dtype=torch.long, device=device)


In [6]:
from transformers import AutoModel
from transformers import AutoTokenizer

BASE_MODEL = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModel.from_pretrained("openai-community/gpt2")
model.save_pretrained(BASE_MODEL)

tokenizer.save_pretrained(BASE_MODEL)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

('openai-community/gpt2/tokenizer_config.json',
 'openai-community/gpt2/special_tokens_map.json',
 'openai-community/gpt2/vocab.json',
 'openai-community/gpt2/merges.txt',
 'openai-community/gpt2/added_tokens.json',
 'openai-community/gpt2/tokenizer.json')

In [12]:
from model import GPTConfig, GPT
import os
import torch

In [28]:
model_type = 'gpt2-large'
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
learning_rate = 6e-4 # max learning rate
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1' etc., or try 'mps' on macbooks
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast


assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
from transformers import GPT2LMHeadModel
print("loading weights from pretrained gpt: %s" % model_type)

# n_layer, n_head and n_embd are determined from model_type
config_args = {
    'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
    'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
    'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
    'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
}[model_type]
config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
config_args['bias'] = True # always True for GPT model checkpoints


# create a from-scratch initialized minGPT model
config = GPTConfig(**config_args)
model = GPT.from_pretrained(model_type)
model_args = {}
for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
    model_args[k] = getattr(model.config, k)
model.to(device)

optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)


checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'model_args': model_args,
    'iter_num': 0,
    'best_val_loss': None,
    'config': config,
}
out_dir = f"output_dump/test_{model_type}_eval"
os.makedirs(out_dir, exist_ok=True)
print(f"saving checkpoint to {out_dir}")
torch.save(checkpoint, os.path.join(out_dir, 'ckpt.pt'))



loading weights from pretrained gpt: gpt2-large
loading weights from pretrained gpt: gpt2-large
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 772.72M


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

num decayed parameter tensors: 146, with 773,428,480 parameters
num non-decayed parameter tensors: 290, with 601,600 parameters
using fused AdamW: True
saving checkpoint to output_dump/test_gpt2-large_eval


In [26]:
from transformers import GPT2TokenizerFast, GPT2Tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
import tiktoken
tokenizer_2 = tiktoken.get_encoding("gpt2")

tokenizer3 = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

bunch_of_random_strings = [
    "Hello, how are you?",
    "I am doing well, thank you for asking.",
    "set the temperature to 72 degrees",
    "what is the meaning of life?",
    "To be, or not to be, that is the question.",
    "I think, therefore I am.",
    "I am groot",
    "No, I am groot",
    "N O T H I S I S"
    "IT is that is"
    "Tyler durden Kino fist"
    "hidsakgfidjhfgdksjfhksdjfhdkjshfk"
    ]
for s in bunch_of_random_strings:
    a = tokenizer(s)["input_ids"]
    b = tokenizer_2.encode(s)
    c = tokenizer3.encode(s)
    #print(a, "\n", b, "\n", a == b)
    print(a, "\n", c, "\n", a == c)


[15496, 11, 703, 389, 345, 30] 
 [15496, 11, 703, 389, 345, 30] 
 True
[40, 716, 1804, 880, 11, 5875, 345, 329, 4737, 13] 
 [40, 716, 1804, 880, 11, 5875, 345, 329, 4737, 13] 
 True
[2617, 262, 5951, 284, 7724, 7370] 
 [2617, 262, 5951, 284, 7724, 7370] 
 True
[10919, 318, 262, 3616, 286, 1204, 30] 
 [10919, 318, 262, 3616, 286, 1204, 30] 
 True
[2514, 307, 11, 393, 407, 284, 307, 11, 326, 318, 262, 1808, 13] 
 [2514, 307, 11, 393, 407, 284, 307, 11, 326, 318, 262, 1808, 13] 
 True
[40, 892, 11, 4361, 314, 716, 13] 
 [40, 892, 11, 4361, 314, 716, 13] 
 True
[40, 716, 7128, 313] 
 [40, 716, 7128, 313] 
 True
[2949, 11, 314, 716, 7128, 313] 
 [2949, 11, 314, 716, 7128, 313] 
 True
[45, 440, 309, 367, 314, 311, 314, 311, 2043, 318, 326, 318, 46807, 288, 42568, 509, 2879, 18606, 71, 2340, 461, 70, 69, 312, 73, 71, 69, 21287, 591, 73, 69, 71, 591, 28241, 69, 31298, 42421, 1477, 69, 74] 
 [45, 440, 309, 367, 314, 311, 314, 311, 2043, 318, 326, 318, 46807, 288, 42568, 509, 2879, 18606, 71, 23

In [17]:
from transformers import GPT2LMHeadModel
import transformers
model_type = 'bbunzeck/gpt-wee-regular'
model_hf = GPT2LMHeadModel.from_pretrained("gpt2")
model2 = transformers.AutoModelForCausalLM.from_pretrained("bbunzeck/gpt-wee-regular")
print(model_hf)
#print(model_hf)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [20]:
from model import GPTConfig, GPT
import os
import torch

model_type = 'bbunzeck/gpt-wee-regular'
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
learning_rate = 6e-4 # max learning rate
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1' etc., or try 'mps' on macbooks
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast


assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl', "bbunzeck/gpt-wee-regular"}
#override_args = override_args or {} # default to empty dict
# only dropout can be overridden see more notes below
#assert all(k == 'dropout' for k in override_args)
from transformers import GPT2LMHeadModel
print("loading weights from pretrained gpt: %s" % model_type)

# n_layer, n_head and n_embd are determined from model_type
config_args = {
    'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
    'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
    'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
    'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
    "bbunzeck/gpt-wee-regular":      dict(n_layer=2, n_head=2, n_embd=128),
}[model_type]
#print("forcing vocab_size=8k, block_size=1024, bias=True")
config_args['vocab_size'] = 8000 # always 50257 for GPT model checkpoints
config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
config_args['bias'] = True # always True for GPT model checkpoints
config_args['dropout'] = 0.1
# we can override the dropout rate, if desired

# create a from-scratch initialized minGPT model
config = GPTConfig(**config_args)
model = GPT(config)
sd = model.state_dict()
sd_keys = sd.keys()
sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

# init a huggingface/transformers model
model_hf = GPT2LMHeadModel.from_pretrained(model_type)
sd_hf = model_hf.state_dict()

# copy while ensuring all of the parameters are aligned and match in names and shapes
sd_keys_hf = sd_hf.keys()
sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
# basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
# this means that we have to transpose these weights when we import them
assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
for k in sd_keys_hf:
    if any(k.endswith(w) for w in transposed):
        # special treatment for the Conv1D weights we need to transpose
        assert sd_hf[k].shape[::-1] == sd[k].shape
        with torch.no_grad():
            sd[k].copy_(sd_hf[k].t())
    else:
        # vanilla copy over the other parameters
        #print(sd_hf[k].shape, sd[k].shape)
        assert sd_hf[k].shape == sd[k].shape
        with torch.no_grad():
            sd[k].copy_(sd_hf[k])

model_args = {}
for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
    model_args[k] = getattr(model.config, k)
model.to(device)

optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)


checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'model_args': model_args,
    'iter_num': 0,
    'best_val_loss': None,
    'config': config,
}
out_dir = f"output_dump/test_GPT_wee_nanogpt_eval"
os.makedirs(out_dir, exist_ok=True)
print(f"saving checkpoint to {out_dir}")
torch.save(checkpoint, os.path.join(out_dir, 'ckpt.pt'))



loading weights from pretrained gpt: bbunzeck/gpt-wee-regular
number of parameters: 1.42M
num decayed parameter tensors: 10, with 1,548,288 parameters
num non-decayed parameter tensors: 18, with 3,584 parameters
using fused AdamW: True
saving checkpoint to output_dump/test_GPT_wee_nanogpt_eval


In [22]:
 batched_inputs = tensor([[   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
          223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
          305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913]],
       device='cuda:0')

# batched_inputs.shape
#torch.Size([1, 36])




NameError: name 'tensor' is not defined

In [ ]:


-> for _, context_enc, continuation_enc in chunk:
(Pdb) input
tensor([   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
         223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
         305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913],
       device='cuda:0')
(Pdb) input.shape
torch.Size([36])
(Pdb) inputs
[tensor([[   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
          223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
          305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913]],
       device='cuda:0')]
(Pdb) inputs.shape
*** AttributeError: 'list' object has no attribute 'shape'
(Pdb) cont_tokens_list
[[39, 1401, 202, 230, 1161, 634, 347, 355, 1611, 341, 196, 223, 508, 554, 782, 7498, 202, 596, 1161, 634, 2084, 479, 932, 305, 176, 49, 225, 749, 63, 735, 196, 2176, 287, 999, 913, 14]]
(Pdb) c
> /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/evaluation_pipeline/lm_eval/api/model.py(319)_loglikelihood_tokens()
-> multi_logits_psoft = self._model_call(batched_inputs)
(Pdb) batched_inputs
tensor([[   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
          223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
          305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913]],
       device='cuda:0')
(Pdb) batched_inputs.shape
torch.Size([1, 36])
(Pdb) self._model_call(batched_inputs)
tensor([[[ 0.9603,  0.0904, -0.2527,  ..., -0.1419, -0.1697, -0.0142],
         [ 0.2586, -0.0471, -0.0341,  ...,  0.0661,  0.0973,  0.1385],
         [ 0.0509, -0.0472, -0.0703,  ..., -0.4754,  0.2393,  0.0844],
         ...,
         [ 0.0344, -0.4969, -0.3445,  ...,  0.2487, -0.0975, -0.0794],
         [ 0.1197,  0.2152, -0.0201,  ..., -0.0680,  0.0684, -0.0025],
         [ 0.0146,  0.4704, -0.0295,  ...,  0.0201, -0.3023, -0.0791]]],
       device='cuda:0')
(Pdb) self._model_call(batched_inputs)[0]
tensor([[ 0.9603,  0.0904, -0.2527,  ..., -0.1419, -0.1697, -0.0142],
        [ 0.2586, -0.0471, -0.0341,  ...,  0.0661,  0.0973,  0.1385],
        [ 0.0509, -0.0472, -0.0703,  ..., -0.4754,  0.2393,  0.0844],
        ...,
        [ 0.0344, -0.4969, -0.3445,  ...,  0.2487, -0.0975, -0.0794],
        [ 0.1197,  0.2152, -0.0201,  ..., -0.0680,  0.0684, -0.0025],
        [ 0.0146,  0.4704, -0.0295,  ...,  0.0201, -0.3023, -0.0791]],
       device='cuda:0')
(Pdb) self._model_call(batched_inputs)[0].shape
torch.Size([36, 8000])
(Pdb) self._model_call(batched_inputs)[0][0]
tensor([ 0.9603,  0.0904, -0.2527,  ..., -0.1419, -0.1697, -0.0142],
       device='cuda:0')
(Pdb) self._model_call(batched_inputs)[0][0].shape
torch.Size([8000])
(Pdb) self._model_call(batched_inputs)[0][0][:10]
tensor([ 0.9603,  0.0904, -0.2527, -0.1411,  0.0711,  0.1061,  0.1519, -0.0959,
         0.3926, -0.0770], device='cuda:0')
(Pdb)


In [ ]:
#HF Model

[(('', " A brother of that waitress compels there to be more than three daughters of every waitress motivating Derek's handyman to tour this high school."), [0], [39, 1401, 202, 230, 1161, 634, 347, 355, 1611, 341, 196, 223, 508, 554, 782, 7498, 202, 596, 1161, 634, 2084, 479, 932, 305, 176, 49, 225, 749, 63, 735, 196, 2176, 287, 999, 913, 14])]

(Pdb)  self._model_call(batched_inputs)
tensor([[[-16.8088,   5.1051,   1.7680,  ...,  -2.4424,  -4.7187,  -1.2489],
         [-15.6280,   5.1829,   0.7787,  ...,  -2.5094,  -7.5187,  -2.0798],
         [-15.8441,   6.0225,   0.9640,  ...,  -0.1196,  -3.7741,  -0.2289],
         ...,
         [-13.2072,   1.9264,   1.5937,  ...,  -3.8293,  -8.7227,  -0.3560],
         [-13.2464,   2.3742,   1.6671,  ...,  -2.8522,  -3.4051,  -3.0457],
         [-16.4980,   3.7981,   0.1934,  ...,  -1.6204,  -5.4572,  -2.1577]]],
       device='cuda:0')

(Pdb) self._model_call(batched_inputs)[0]
tensor([[-16.8088,   5.1051,   1.7680,  ...,  -2.4424,  -4.7187,  -1.2489],
        [-15.6280,   5.1829,   0.7787,  ...,  -2.5094,  -7.5187,  -2.0798],
        [-15.8441,   6.0225,   0.9640,  ...,  -0.1196,  -3.7741,  -0.2289],
        ...,
        [-13.2072,   1.9264,   1.5937,  ...,  -3.8293,  -8.7227,  -0.3560],
        [-13.2464,   2.3742,   1.6671,  ...,  -2.8522,  -3.4051,  -3.0457],
        [-16.4980,   3.7981,   0.1934,  ...,  -1.6204,  -5.4572,  -2.1577]],
       device='cuda:0')
(Pdb) self._model_call(batched_inputs)[0].shape
torch.Size([36, 8000])
(Pdb) self._model_call(batched_inputs)[0][0]
tensor([-16.8088,   5.1051,   1.7680,  ...,  -2.4424,  -4.7187,  -1.2489],
       device='cuda:0')
(Pdb) self._model_call(batched_inputs)[0][0].shape
torch.Size([8000])
(Pdb) self._model_call(batched_inputs)[0][0][:10]
tensor([-16.8088,   5.1051,   1.7680,  -2.3771,  -3.8390,  -6.4859,  -3.6476,
          1.1271,  -0.4308,   0.1051], device='cuda:0')


(Pdb) multi_logits
tensor([[[-26.6113,  -4.6974,  -8.0345,  ..., -12.2449, -14.5212, -11.0514],
         [-25.0941,  -4.2832,  -8.6874,  ..., -11.9755, -16.9848, -11.5459],
         [-25.9500,  -4.0834,  -9.1420,  ..., -10.2256, -13.8800, -10.3348],
         ...,
         [-23.3123,  -8.1786,  -8.5114,  ..., -13.9344, -18.8278, -10.4611],
         [-23.7417,  -8.1211,  -8.8282,  ..., -13.3476, -13.9004, -13.5410],
         [-26.7012,  -6.4051, -10.0098,  ..., -11.8236, -15.6604, -12.3608]]])


(Pdb) greedy_tokens
tensor([[  12,   12,   12,  173,   14,  181,   14,   56, 2123,   14,   14,  223,
          171,  554,  171,  763,   14,  173,  599,  340,   14,  479,  300,   14,
         6025,   14,   14, 1497,   14,   14,   14,  173, 1258,  971,  913,   14]])
(Pdb)

tensor([[ -6.7429,  -9.8875,  -3.7845,  -6.2735, -11.1070,  -3.1127,  -8.7543,
          -1.8242,  -2.2615,  -6.6816,  -4.0765,  -2.0897,  -4.2425,  -1.4785,
          -3.9279,  -5.5701,  -2.3604,  -7.0383,  -8.9980,  -2.3753, -11.5858,
          -1.3071,  -2.3076,  -7.2952,  -4.4967,  -2.9648,  -4.0369,  -6.1667,
          -4.5349,  -5.7607,  -3.7400,  -9.0784,  -6.9577,  -7.4068,  -0.8508,
          -1.0929]])

(Pdb) (float(logits.sum()), bool(max_equal))
(-182.16981506347656, False)


In [7]:
from transformers import GPT2LMHeadModel
import torch
from model import GPTConfig, GPT
import transformers 

model_ngpt_path = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/test_gpt2_eval/ckpt.pt'
#model_ngpt = torch.load(model_ngpt_path)

device = 'cuda'
checkpoint = torch.load(model_ngpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
print("G2",gptconf)
model_ngpt = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model_ngpt.load_state_dict(state_dict)

model_ngpt.to(device)
model_ngpt.eval()

from transformers import AutoModel
from transformers import AutoTokenizer

BASE_MODEL = "openai-community/gpt2"

tokenizer_hf = AutoTokenizer.from_pretrained(BASE_MODEL)
model_hf = transformers.AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
model_hf.to(device)
model_hf.eval()
#model_hf = GPT2LMHeadModel.from_pretrained("gpt2")



G2 GPTConfig(block_size=1024, vocab_size=50257, n_layer=12, n_head=12, n_embd=768, dropout=0.0, bias=True, wm_mask=False, wm_decay_rate=1, wm_decay_type='linear')
number of parameters: 123.65M


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
# input_array = torch.tensor([   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
#           223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
#           305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913])

input_array = torch.tensor([   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
          223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
          305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913])
input_array = input_array.unsqueeze(0).to(device)
input_array.shape

torch.Size([1, 36])

In [4]:
input_array

tensor([[   0,   39, 1401,  202,  230, 1161,  634,  347,  355, 1611,  341,  196,
          223,  508,  554,  782, 7498,  202,  596, 1161,  634, 2084,  479,  932,
          305,  176,   49,  225,  749,   63,  735,  196, 2176,  287,  999,  913]],
       device='cuda:0')

In [5]:
model_hf(input_array)["logits"]



NameError: name 'model_hf' is not defined

In [11]:
model_ngpt(input_array, input_array)[0]

tensor([[[-37.6848, -38.1988, -40.4196,  ..., -47.3130, -46.7274, -36.7948],
         [-74.2754, -75.0781, -75.2163,  ..., -85.6687, -82.1121, -74.6503],
         [-71.1247, -71.7637, -72.3571,  ..., -77.2431, -78.9723, -73.2625],
         ...,
         [-72.9447, -72.2903, -74.1882,  ..., -77.0912, -77.6301, -72.8227],
         [-82.0377, -81.7375, -84.4531,  ..., -86.7854, -86.5893, -81.4120],
         [-83.3041, -82.2193, -84.9484,  ..., -86.8005, -88.0736, -81.9391]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [39]:
from transformers import GPT2LMHeadModel
import torch
from model import GPTConfig, GPT
import transformers 

model_ngpt_path = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/test_GPT_wee_nanogpt_eval/ckpt.pt'
#model_ngpt = torch.load(model_ngpt_path)

device = 'cuda'
checkpoint = torch.load(model_ngpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
print("G2",gptconf)
model_ngpt = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model_ngpt.load_state_dict(state_dict)

model_ngpt.to(device)


from transformers import AutoModel
from transformers import AutoTokenizer

BASE_MODEL = "openai-community/gpt2"

tokenizer_hf = AutoTokenizer.from_pretrained(BASE_MODEL)
model_hf = transformers.AutoModelForCausalLM.from_pretrained("bbunzeck/gpt-wee-regular")
model_hf.to(device)
#model_hf = GPT2LMHeadModel.from_pretrained("gpt2")



G2 GPTConfig(block_size=1024, vocab_size=8000, n_layer=2, n_head=2, n_embd=128, dropout=0.0, bias=True, wm_mask=False, wm_decay_rate=1, wm_decay_type='linear')
number of parameters: 1.42M


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(8000, 128)
    (wpe): Embedding(1024, 128)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=128, out_features=8000, bias=False)
)

In [40]:
model_hf(input_array)["logits"]

tensor([[[-16.8088,   5.1051,   1.7680,  ...,  -2.4424,  -4.7187,  -1.2489],
         [-15.6280,   5.1829,   0.7787,  ...,  -2.5094,  -7.5187,  -2.0798],
         [-15.8441,   6.0225,   0.9640,  ...,  -0.1196,  -3.7741,  -0.2289],
         ...,
         [-13.2072,   1.9264,   1.5937,  ...,  -3.8293,  -8.7227,  -0.3560],
         [-13.2464,   2.3742,   1.6671,  ...,  -2.8522,  -3.4051,  -3.0457],
         [-16.4980,   3.7981,   0.1934,  ...,  -1.6204,  -5.4572,  -2.1577]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [41]:
model_ngpt(input_array, input_array)[0]

tensor([[[-16.8098,   5.1036,   1.7663,  ...,  -2.4418,  -4.7190,  -1.2489],
         [-15.6291,   5.1829,   0.7782,  ...,  -2.5109,  -7.5197,  -2.0793],
         [-15.8458,   6.0229,   0.9636,  ...,  -0.1206,  -3.7752,  -0.2286],
         ...,
         [-13.2081,   1.9281,   1.5941,  ...,  -3.8275,  -8.7236,  -0.3567],
         [-13.2485,   2.3732,   1.6661,  ...,  -2.8536,  -3.4064,  -3.0453],
         [-16.5025,   3.7976,   0.1941,  ...,  -1.6214,  -5.4600,  -2.1579]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [2]:
from transformers import GPT2LMHeadModel
import torch
from model import GPTConfig, GPT
import transformers 

model_ngpt_path = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-2x2-nomask-1709513019/ckpt.pt'
#model_ngpt = torch.load(model_ngpt_path)

device = 'cuda'
checkpoint = torch.load(model_ngpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
print("G2",gptconf)
model_ngpt = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model_ngpt.load_state_dict(state_dict)

model_ngpt.to(device)
model_ngpt.eval()

# from transformers import AutoModel
# from transformers import AutoTokenizer

# BASE_MODEL = "openai-community/gpt2"

# tokenizer_hf = AutoTokenizer.from_pretrained(BASE_MODEL)
# model_hf = transformers.AutoModelForCausalLM.from_pretrained("bbunzeck/gpt-wee-regular")
# model_hf.to(device)
# #model_hf = GPT2LMHeadModel.from_pretrained("gpt2")



G2 GPTConfig(block_size=128, vocab_size=8000, n_layer=2, n_head=2, n_embd=128, dropout=0.1, bias=False, wm_mask=False, wm_decay_rate=1, wm_decay_type='linear')
number of parameters: 1.42M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8000, 128)
    (wpe): Embedding(128, 128)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=128, out_features=384, bias=False)
          (c_proj): Linear(in_features=128, out_features=128, bias=False)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=512, out_features=128, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=128, out_features=8000, bias=False)
)

In [6]:
model_ngpt(input_array, input_array)[0]

tensor([[[-20.3146,  -0.0394,  -2.0770,  ...,  -6.3831, -12.1462,  -9.5530],
         [-15.8794,  -1.4475,  -0.2534,  ...,  -6.6624,  -9.9852,  -5.0226],
         [-17.5125,  -0.9478,  -0.8446,  ...,  -5.3582,  -9.4216,  -4.0392],
         ...,
         [-19.8940,  -2.6152,  -2.5639,  ..., -10.3370, -10.9929,  -6.1031],
         [-15.7436,  -1.7505,  -2.0812,  ...,  -6.3878,  -8.9448,  -3.4779],
         [-21.9473,  -1.3997,  -3.7890,  ...,  -8.1338, -10.6655,  -7.6561]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [1]:
import torch
import os
import numpy as np
from model import GPTConfig, GPT

out_dir = "output_dump/out-babylm_full_bpe_8k-6x6-mask_e010-6168468_nm"
device = "cuda"
dataset = "babylm_full_bpe_8k"
data_dir = os.path.join('data', dataset)
print(f"loading data from {data_dir}")
train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')
block_size = 256
batch_size = 32
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast

print(f"Resuming training from {out_dir}")
# resume training from a checkpoint.
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
checkpoint_model_args = checkpoint['model_args']
# force these config attributes to be equal otherwise we can't even resume training
# the rest of the attributes (e.g. dropout) can stay as desired from command line
# for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
#     model_args[k] = checkpoint_model_args[k]
# create the model
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
# fix the keys of the state dictionary :(
# honestly no idea how checkpoints sometimes get this prefix, have to debug more
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
iter_num = checkpoint['iter_num']
print("continuing from iteration", iter_num)
best_val_loss = checkpoint['best_val_loss']
model.to(device)


def get_batch(split, batchloader='random', batch_idx = 0):

    data = train_data if split == 'train' else val_data
    if batchloader == 'random':
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
        if device_type == 'cuda':
            # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
            x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
        else:
            x, y = x.to(device), y.to(device)
        return x, y

    elif batchloader == 'full':
        pass

X, Y = get_batch('train')
print(best_val_loss)

logits, loss = model(X, Y)

print("hr")
print(logits.shape, loss)

loading data from data/babylm_full_bpe_8k
Resuming training from output_dump/out-babylm_full_bpe_8k-6x6-mask_e010-6168468_nm
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 13.69M
continuing from iteration 44000
tensor(0.0387, device='cuda:0')
idx:  tensor([[ 171,  844,  179,  ...,   13,  238,  733],
        [ 591,  192,  263,  ..., 1494, 1204,  207],
        [5290, 7775,   14,  ...,  290,  230, 1399],
        ...,
        [ 903,  239, 1347,  ...,  133,  242,  173],
        [4703,  975,  960,  ...,   12,  232, 2263],
        [ 620, 1708,  292,  ...,  253,   41,  670]], device='cuda:0')
targets shape:  torch.Size([32, 256])
logits shape:  torch.Size([32, 256, 8000])
reshaped targets:  torch.Size([8192])
reshaped logit

In [42]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch.nn.functional as F

# Load the pretrained GPT-2 model and tokenizer
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Ensure the model is in evaluation mode
model.eval()

# Define a function to calculate surprisal
def calculate_surprisal(text, model, tokenizer):
    # Tokenize the input text
    token_ids = tokenizer.encode(text, return_tensors='pt')
    #append BOS token
    token_ids = torch.cat([torch.tensor([[tokenizer.bos_token_id]]), token_ids], dim=1)
    # Get the logits from the model
    with torch.no_grad():
        outputs = model(token_ids)
        logits = outputs.logits

    # Convert logits to log probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # Calculate the negative log probability (surprisal) for each token
    surprisals = []
    for i in range(1, len(token_ids[0])):
        token_log_prob = log_probs[0, i-1, token_ids[0, i]].item()
        surprisals.append(-token_log_prob)

    return surprisals

# Example usage
text = "I told"
surprisals = calculate_surprisal(text, model, tokenizer)
#print(f"Surprisals for '{text}': {surprisals}")
# Print the surprisal for each token in the input text
for token, surprisal in zip(tokenizer.tokenize(text), surprisals):
    print(f"{token}: {surprisal:.2f}")

I: 4.00
Ġtold: 6.86


In [45]:
def return_surprisals(model, token_list):
    # if len(token_list)>model.config.block_size:
    #     token_list = token_list[-model.config.block_size:]
    token_tensor = torch.tensor(token_list)#.unsqueeze(0)
    #print(token_tensor.shape)
    with torch.no_grad():
        #logits, _ = model(token_tensor)
        logits = model(token_tensor[:-1]).logits
        #print(logits)

    #logits = logits[0, -1, :]
    probs = F.log_softmax(logits, dim=-1)
    print(token_list[-1])
    token_logprob = probs[-1, token_list[-1]].item()
    return -token_logprob


text = "I told you the cat is on the mat"
token_list = tokenizer.encode(text)
token_list = [tokenizer.bos_token_id] + token_list
print(token_list)

surprisals = []

for i in range(2, len(token_list)+1):
    surprisal = return_surprisals(model, token_list[:i])
    surprisals.append(surprisal)

for token, surprisal in zip(tokenizer.tokenize(text), surprisals):
    print(f"{token}: {surprisal:.2f}")



[50256, 40, 1297, 345, 262, 3797, 318, 319, 262, 2603]
40
1297
345
262
3797
318
319
262
2603
I: 4.00
Ġtold: 6.86
Ġyou: 0.62
Ġthe: 4.12
Ġcat: 7.61
Ġis: 3.03
Ġon: 4.82
Ġthe: 1.23
Ġmat: 7.03


In [50]:
def return_surprisals(model, token_list):

    token_tensor = torch.tensor(token_list)
    with torch.no_grad():
        logits = model(token_tensor[:-1]).logits
    #logits = logits[:-1, :]
    probs = F.log_softmax(logits, dim=-1)
    token_logprob = probs[-1, token_list[-1]].item()
    return -token_logprob

text = "I told you the cat is on the mat"
token_list = tokenizer.encode(text)
token_list = [tokenizer.bos_token_id] + token_list
print(token_list)

surprisals = []

for i in range(2, len(token_list)+1):
    surprisal = return_surprisals(model, token_list[:i])
    surprisals.append(surprisal)

for token, surprisal in zip(tokenizer.tokenize(text), surprisals):
    print(f"{token}: {surprisal:.2f}")

[50256, 40, 1297, 345, 262, 3797, 318, 319, 262, 2603]
I: 4.00
Ġtold: 6.86
Ġyou: 0.62
Ġthe: 4.12
Ġcat: 7.61
Ġis: 3.03
Ġon: 4.82
Ġthe: 1.23
Ġmat: 7.03


In [34]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch.nn.functional as F

# Load the pretrained GPT-2 model and tokenizer
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Ensure the model is in evaluation mode
model.eval()

# Define a function to calculate surprisal step-by-step
def calculate_surprisal_step_by_step(text, model, tokenizer):
    token_ids = tokenizer.encode(text, return_tensors='pt')[0]
    token_ids = torch.cat([torch.tensor([tokenizer.bos_token_id]), token_ids], dim=0)
    surprisals = []
    for i in range(1, len(token_ids)):
        # Get the logits for the current context (up to the current token)
        with torch.no_grad():
            outputs = model(token_ids[:i].unsqueeze(0))
            logits = outputs.logits

        # Get the log probability of the next token
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_prob = log_probs[0, -1, token_ids[i]].item()
        surprisals.append(-token_log_prob)

    return surprisals

# Example usage
text = "I told you the cat is on the mat"
surprisals = calculate_surprisal_step_by_step(text, model, tokenizer)
for token, surprisal in zip(tokenizer.tokenize(text), surprisals):
    print(f"{token}: {surprisal:.2f}")


I: 4.00
Ġtold: 6.86
Ġyou: 0.62
Ġthe: 4.12
Ġcat: 7.61
Ġis: 3.03
Ġon: 4.82
Ġthe: 1.23
Ġmat: 7.03


In [38]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch.nn.functional as F

# Load the pretrained GPT-2 model and tokenizer
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Ensure the model is in evaluation mode
model.eval()

# Define a function to calculate surprisal
def calculate_surprisal(text, model, tokenizer, max_length):
    token_ids = tokenizer.encode(text, return_tensors='pt')[0]

    surprisals = []
    start = 0
    while start < len(token_ids):
        end = min(start + max_length, len(token_ids))
        chunk_ids = token_ids[start:end]

        # Get the logits for the current chunk
        with torch.no_grad():
            outputs = model(chunk_ids.unsqueeze(0))
            logits = outputs.logits

        # Convert logits to log probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Calculate surprisal for the current chunk
        if start == 0:
            # First chunk: start from the second token
            range_start = 1
        else:
            # Subsequent chunks: use the entire chunk
            range_start = 0

        for i in range(range_start, len(chunk_ids)):
            token_log_prob = log_probs[0, i - 1, chunk_ids[i]].item()
            surprisals.append(-token_log_prob)

        # Move the start to the next position
        start = end - max_length + 1  # Overlap by max_length - 1 tokens

    return surprisals

# Example usage
text = "This is a very long sentence that exceeds the maximum context length of the model, so we need to handle it by chunking the text appropriately."
max_length = model.config.n_positions  # GPT-2's maximum context length
surprisals = calculate_surprisal(text, model, tokenizer, max_length)
print(f"Surprisals for '{text}': {surprisals}")


KeyboardInterrupt: 